# 🫀 실험 13 — 우리 자를 **아는 답**에 대본다 (소견 평면 대조)

**MedKOS / `notebooks/exp13_finding_plane_control.ipynb`** · 퀘스트 `ailab-2026-0015`
**학습 0회** — 실험10′의 out-of-fold 확률을 다시 자르기만 한다.

---

## 이건 발견 실험이 아니라 **계측기 검증**이다

"하벽경색은 II·III·aVF, 전중격경색은 V1–V4"는 **의대 본과 1학년이 아는 사실**이다.
그런데 그게 **바로 이 실험의 조건**이다. 답을 모르는 문제로는 자를 검증할 수 없다.

이 퀘스트 전체가 하나의 주장 위에 서 있다:

> **유도군별 성능 차이는 그 소견이 어느 해부학적 평면에 사는지를 측정한다.**

이 주장은 **한 번도 정답과 대조된 적이 없다.** 현재 성적표는 이렇다:

| | 결과 |
|---|---|
| 혼합군(MI) 교호작용 | ✅ +0.0859 [+0.0672, +0.1052] — 확증 |
| 횡단면군(HYP) 교호작용 | ❌ 예측 실패 → Cornell voltage로 **사후 설명** |
| HYP 쪼개서 재검(실험10″) | ⚠️ 표본 없음 — 확인도 반증도 못 함 |

**사후 설명이 하나 끼어 있는 성적표다.** Cornell 이야기가 옳을 수도 있지만, 결과를 보고
만든 이야기라는 사실은 변하지 않는다. 이 상태로 차별점 A(라벨군별 유도 이득 표)를
논문이나 규제 문서에 쓰는 건 위험하다.

**그래서 정답이 확실한 문제를 하나 풀린다.** 통과하면 자가 교정된 것이고, 실패하면
**실험10·10′·10″와 차별점 A가 통째로 재검토 대상**이 된다. 그게 이 실험의 값이다.

## 왜 MI인가 — HYP에 없던 두 가지가 있다

| | HYP | **MI** |
|---|---|---|
| 양성 대조(전두면 소견) | 없음 | **IMI 하벽경색** — II·III·aVF |
| 음성 대조(횡단면 소견) | RVH **2건** | **ASMI/AMI 전중격경색** — V1–V4 |
| 표본 | 535건, 85%가 LVH | **2,532건**, 여러 부위로 갈림 |

해부학적 근거가 교과서이고, 두 군 모두 표본이 수백 건대라 **검정력이 있다.**

## 사전등록 (결과 보기 전에 고정)

| | 예측 | 성격 |
|---|---|---|
| **G0 검정력** | 순수 소집단 n ≥ 50 인 것만 판정 | 10″에서 n=40이면 CI 폭이 1.0이었다 → 문턱을 올린다 |
| **G-기준선** | **MI 전체의 비중을 같은 지표로 같은 실행에서 계산**해 표에 넣는다 | **10″의 교훈** — 비중은 지표 불변량이 아니다. 다른 지표로 잰 65%에 대는 실수를 반복하지 않는다 |
| **P-A ★핵심** | 비중(하벽) − 비중(전벽·중격) **> 0**, CI가 0 제외 | 계측기 검증의 본체 |
| **P-B** | 비중(전벽·중격) **< 33%** | 진짜 음성 대조 |
| **P-C** | 비중(하벽) **> 기준선(MI 전체)** | 절대값이 아니라 **같은 지표 기준선** 대비 (10″ 교훈 반영) |
| **N0 무효 대조** | 하벽군을 **무작위 반으로 갈라** 두 반쪽의 비중 차를 잰다 → CI가 0을 포함해야 한다 | CI 기계가 없는 차이를 만들어내는지 확인. **통과는 증거가 아니고 실패만 경보**다 |

**P-A가 실패하면 그게 이 실험의 결론이다** — 자가 평면을 못 재는 것이므로,
실험10′의 혼합군 확증(+0.0859)조차 다른 이유로 나온 값일 수 있다고 봐야 한다.

## 부수 검증 — STTC로 한 번 더

같은 대조가 허혈에도 있다: **ISCI(하벽 허혈, 전두면)** vs **ISCA(전벽 허혈, 횡단면)**.
표본이 되면 같은 예측을 반복한다. **독립 복제**라 P-A와 방향이 같으면 훨씬 강해진다.

## 지표 (10″와 동일 — 바꾸지 않는다)

- **주지표**: 소집단 AUC = 그 소집단의 해당 superclass 확률이 무작위 NORM보다 높을 확률
- **부지표**: 동작점 정합 재현율 (실험10′ 헤드라인과 잇기 위해)
- 비중 = `[AUC(I+II) − AUC(II)] / [AUC(12) − AUC(II)]`, 분모 CI가 0을 포함하면 **정의불가**
- 판정은 **지지 / 기각 / 미결** 3분. CI가 임계값을 걸치면 기각이 아니라 미결


In [ ]:
# CELL 1 — 설정 (학습 없음)
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

CLASSES = ["NORM", "CD", "STTC", "MI", "HYP"]
CONFIGS = {"II": [1], "I+II": [0, 1], "II+V1": [1, 6], "12": list(range(12))}
K_FOLD, SEED0, BOOT, NMIN = 5, 20260801, 4000, 50

# ── 소견을 해부학적 평면으로 묶는다 (PTB-XL diagnostic_subclass)
#    근거는 '그 소견을 읽는 유도가 어느 평면인가' 하나뿐이다.
PANELS = {
  "MI": {"cls": "MI", "groups": {
      "하벽(전두면)":        ["IMI"],                    # II · III · aVF
      "전벽·중격(횡단면)":   ["ASMI", "AMI"],            # V1–V4
      "하측벽(혼합)":        ["ILMI"],                   # II·III·aVF + V5·V6
      "측벽(혼합)":          ["LMI", "ALMI"],            # I·aVL + V5·V6
      "후벽(횡단면·상반)":   ["PMI", "IPMI", "IPLMI"],   # V1–V3 상반변화
  }, "pos": "하벽(전두면)", "neg": "전벽·중격(횡단면)"},
  "STTC": {"cls": "STTC", "groups": {
      "하벽 허혈(전두면)":   ["ISCI"],
      "전벽 허혈(횡단면)":   ["ISCA"],
  }, "pos": "하벽 허혈(전두면)", "neg": "전벽 허혈(횡단면)"},
}

CONFIG = dict(exp="exp13_finding_plane_control", quest="ailab-2026-0015",
              parent_exp="exp10p_lead_cv", training="없음(재분석)",
              purpose="유도군 지표가 해부학적 평면을 재는지 '아는 답'으로 검증",
              primary_metric="소집단 AUC 기준 {I,II} 이득 비중",
              predictions={"G0": f"순수 소집단 n>={NMIN}만 판정",
                           "G-기준선": "superclass 전체 비중을 같은 지표로 같은 실행에서 계산",
                           "P-A": "비중(하벽) - 비중(전벽·중격) > 0, CI가 0 제외 [핵심]",
                           "P-B": "비중(전벽·중격) < 0.33",
                           "P-C": "비중(하벽) > 기준선(superclass 전체)",
                           "N0": "하벽군 무작위 반분 비중 차의 CI가 0 포함(실패만 경보)"},
              falsification="P-A 실패 시 실험10·10′·10″와 차별점 A를 재검토 대상으로 내린다",
              panels={k: v["groups"] for k, v in PANELS.items()},
              k_fold=K_FOLD, boot=BOOT, seed0=SEED0, nmin=NMIN)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp13_finding_plane", CONFIG, project=PROJECT)

REG = os.path.join(PROJECT, "registry.jsonl")
prev = None
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") == "exp10p_lead_cv" and os.path.isdir(r.get("dir", "")):
        prev = r
if prev is None:
    raise RuntimeError("registry.jsonl에서 exp10p_lead_cv를 못 찾았습니다")
PREV = prev["dir"]
run.log(f"실험10′ 산출물: {PREV}")

def prev_arm(name):
    p = os.path.join(PREV, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

missing = [f"fixed_{c}_f{k}" for c in CONFIGS for k in range(K_FOLD)
           if prev_arm(f"fixed_{c}_f{k}") is None]
if missing:
    raise RuntimeError(f"실험10′ arm 없음: {missing[:5]} … 총 {len(missing)}개")
run.log(f"✅ 유도고정 arm {len(CONFIGS)*K_FOLD}개 확인 — 학습하지 않습니다")

In [ ]:
# CELL 2 — 라벨만 (X 미사용) + OOF 복원
import pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
for f in ("ptbxl_database.csv", "scp_statements.csv"):
    d = os.path.join(PTB, f)
    if not (os.path.exists(d) and os.path.getsize(d) > 0):
        subprocess.run(["wget", "-q", "-O", d, f"{BASE}/{f}"])
df  = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")
scp = pd.read_csv(os.path.join(PTB, "scp_statements.csv"), index_col=0)

agg    = scp[scp.diagnostic == 1].diagnostic_class.to_dict()
sub_of = scp[scp.diagnostic == 1].diagnostic_subclass.to_dict()
df["sc"] = df.scp_codes.apply(
    lambda s: sorted({agg[k] for k in ast.literal_eval(s) if k in agg}))
sub = df[df.sc.apply(lambda s: len(s) == 1 and s[0] in CLASSES)].copy()

CACHE = run.data("ptbxl_12lead_full.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"캐시가 없습니다: {CACHE}")
z = np.load(CACHE, allow_pickle=True)
Y, FOLD10, EID = z["y"], z["fold"], z["eid"]       # ★ z["X"] 미사용
if len(Y) != len(sub) or not np.array_equal(np.sort(EID), np.sort(sub.index.values)):
    raise RuntimeError("캐시가 지금 sub와 다르다")
sub = sub.loc[EID]
sub["subcls"] = sub.scp_codes.apply(
    lambda s: sorted({sub_of[k] for k in ast.literal_eval(s) if k in sub_of}))
run.log(f"레코드 {len(Y):,} · 클래스 {np.bincount(Y, minlength=5).tolist()} ({CLASSES})")

OOF = {}
for c in CONFIGS:
    arr = np.zeros((len(Y), len(CLASSES)))
    for k in range(K_FOLD):
        arr[np.where((FOLD10 - 1) % K_FOLD == k)[0]] = prev_arm(f"fixed_{c}_f{k}")
    OOF[c] = arr
norm_idx = np.where(Y == 0)[0]
run.log(f"OOF 복원 완료 · NORM 대조 {len(norm_idx):,}건")

# 실험10′과 같은 동작점 정합 (부지표용)
def alpha_for(prob, target):
    lo, hi = 0.02, 50.0
    for _ in range(40):
        mid = (lo * hi) ** 0.5
        p = prob.copy(); p[:, 0] *= mid
        if float((p.argmax(1)[Y == 0] != 0).mean()) > target: lo = mid
        else: hi = mid
    return hi

ref_fa = float((OOF["II"].argmax(1)[Y == 0] != 0).mean())
PRED = {}
for c in CONFIGS:
    p = OOF[c].copy(); p[:, 0] *= alpha_for(OOF[c], ref_fa)
    PRED[c] = p.argmax(1)
run.log(f"기준 오경보율 = 유도고정 {{II}}의 {ref_fa:.3f} (실험10′·10″과 동일 절차)")

### CELL 3 — 【G0】 소집단 인구조사

실험10″의 교훈: **AUC를 보기 전에 센다.** 표본이 없으면 그 자체가 결론이고, 없는 검정력을
있는 척하면 안 된다. 문턱은 10″의 `n≥30`에서 **`n≥50`으로 올렸다** — 실측상 n=40이면
비중 CI 폭이 1.0 근처였기 때문이다.


In [ ]:
# CELL 3 — 소집단 인구조사
CENSUS, MEMB = {}, {}
for pname, P in PANELS.items():
    ci = CLASSES.index(P["cls"])
    idx = np.where(Y == ci)[0]
    codes_seen = sorted({c for i in idx for c in sub.subcls.iloc[i]})
    run.log(f"\n── {pname} superclass {len(idx):,}건 · 관측 subclass {codes_seen}")
    MEMB[pname] = {"__전체__": idx}
    CENSUS[pname] = {"__전체__": {"순수": int(len(idx)), "포함": int(len(idx))}}
    run.log(f"  {'소집단':<22}{'순수':>7}{'포함':>7}   판정")
    run.log(f"  {'[기준선] 전체':<22}{len(idx):>7}{len(idx):>7}   ✅ 기준선")
    for g, codes in P["groups"].items():
        inc  = np.array([bool(set(sub.subcls.iloc[i]) & set(codes)) for i in idx])
        pure = np.array([len(sub.subcls.iloc[i]) > 0
                         and set(sub.subcls.iloc[i]) <= set(codes) for i in idx]) & inc
        MEMB[pname][g] = idx[pure]
        CENSUS[pname][g] = {"순수": int(pure.sum()), "포함": int(inc.sum())}
        run.log(f"  {g:<22}{pure.sum():>7}{inc.sum():>7}   "
                + ("✅ 판정" if pure.sum() >= NMIN else f"⚠️ 서술만 (n<{NMIN})"))

def testable(pname, g): return CENSUS[pname][g]["순수"] >= NMIN

for pname, P in PANELS.items():
    ok = testable(pname, P["pos"]) and testable(pname, P["neg"])
    run.log(f"\n{pname} 패널 P-A 판정 가능 → {'✅' if ok else '⚠️ 아니오'}")

In [ ]:
# CELL 4 — 소집단별 AUC · 비중
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

def auc_fast(sp, sn):
    """Mann-Whitney 항등식. sklearn과 동일하되 ~4배 빠르다(부트스트랩 4천 × 수십 군)."""
    r = rankdata(np.concatenate([sp, sn]), method="average")
    npos, nneg = len(sp), len(sn)
    return float((r[:npos].sum() - npos * (npos + 1) / 2) / (npos * nneg))

# 자체검산: 동점이 많은 재표본에서도 sklearn과 같은가
_rs = np.random.RandomState(0); _a = _rs.rand(200); _b = _rs.rand(2000)
_a, _b = _rs.choice(_a, 200), _rs.choice(_b, 2000)      # 중복 → 동점 유발
_d = abs(auc_fast(_a, _b) - roc_auc_score(
    np.r_[np.ones(200), np.zeros(2000)], np.r_[_a, _b]))
assert _d < 1e-12, f"auc_fast가 sklearn과 다르다 ({_d:.2e})"
run.log(f"auc_fast 자체검산 통과 (sklearn과 오차 {_d:.1e})")

def auc_of(ci, cfg, gidx, gi=None, ni=None):
    g = gidx if gi is None else gidx[gi]
    n = norm_idx if ni is None else norm_idx[ni]
    return auc_fast(OOF[cfg][g, ci], OOF[cfg][n, ci])

q = lambda v, p: float(np.percentile(v, p)) if len(v) else float("nan")

def measure(ci, gidx, seed=SEED0):
    """비중 점추정 + 부트스트랩 표본.

    ★ NORM 대조의 재표본 축을 **모든 군이 공유**해야 군간 비교가 짝지어진다.
      군 인덱스와 NORM 인덱스를 한 RNG에서 번갈아 뽑으면 군 크기가 다를 때
      스트림이 어긋나 NORM 축이 달라진다 → RNG를 둘로 분리한다.
    """
    a = {c: auc_of(ci, c, gidx) for c in CONFIGS}
    d_i2, d_12 = a["I+II"] - a["II"], a["12"] - a["II"]
    rs_g = np.random.RandomState(seed)          # 군마다 독립(표본이 다르므로)
    rs_n = np.random.RandomState(seed + 1)      # NORM은 모든 군에서 동일 스트림
    bs_den, bs_share = [], []
    for _ in range(BOOT):
        gi = rs_g.randint(0, len(gidx), len(gidx))
        ni = rs_n.randint(0, len(norm_idx), len(norm_idx))
        b = {c: auc_of(ci, c, gidx, gi, ni) for c in ("II", "I+II", "12")}
        den = b["12"] - b["II"]
        bs_den.append(den)
        bs_share.append((b["I+II"] - b["II"]) / den if den > 1e-6 else np.nan)
    bs_share = np.array(bs_share); den_ci = (q(bs_den, 2.5), q(bs_den, 97.5))
    den_ok = den_ci[0] > 0
    return {"auc": a, "delta_i2": d_i2, "delta_12": d_12,
            "delta_12_ci": list(den_ci), "denominator_positive": bool(den_ok),
            "share": (d_i2 / d_12) if den_ok and abs(d_12) > 1e-6 else None,
            "share_ci": [q(bs_share[~np.isnan(bs_share)], 2.5),
                         q(bs_share[~np.isnan(bs_share)], 97.5)] if den_ok else None,
            "_bs": bs_share,
            "recall_matched": {c: float((PRED[c][gidx] == ci).mean()) for c in CONFIGS}}

RESULT = {}
t0 = time.time()
for pname, P in PANELS.items():
    ci = CLASSES.index(P["cls"])
    RESULT[pname] = {}
    run.log("\n" + "=" * 104)
    run.log(f"【{pname} 패널】 소집단별 AUC ({P['cls']} 확률 · NORM 대조)")
    run.log("=" * 104)
    run.log(f"  {'소집단':<22}{'n':>6}" + "".join(f"{c:>9}" for c in CONFIGS)
            + f"{'Δ(I+II)':>10}{'Δ(12)':>9}{'비중':>20}")
    for g in ["__전체__"] + list(P["groups"]):
        gidx = MEMB[pname][g]
        if len(gidx) < 5:
            run.log(f"  {g:<22}{len(gidx):>6}   (표본 부족 — 생략)"); continue
        m = measure(ci, gidx)
        m["n"] = int(len(gidx))
        m["testable"] = bool(g == "__전체__" or testable(pname, g))
        RESULT[pname][g] = m
        s = (f"{m['share']:>7.1%} [{m['share_ci'][0]:.0%},{m['share_ci'][1]:.0%}]"
             if m["share"] is not None else "   정의불가(분모 CI 0 포함)")
        tag = "" if m["testable"] else "  ⚠️서술만"
        label = "[기준선] 전체" if g == "__전체__" else g
        run.log(f"  {label:<22}{len(gidx):>6}" + "".join(f"{m['auc'][c]:>9.3f}" for c in CONFIGS)
                + f"{m['delta_i2']:>+10.3f}{m['delta_12']:>+9.3f}{s:>20}{tag}")

    run.log(f"\n  【부지표】 동작점 정합 {P['cls']} 재현율")
    run.log(f"  {'소집단':<22}" + "".join(f"{c:>9}" for c in CONFIGS))
    for g, m in RESULT[pname].items():
        label = "[기준선] 전체" if g == "__전체__" else g
        run.log(f"  {label:<22}" + "".join(f"{m['recall_matched'][c]:>9.3f}" for c in CONFIGS))
run.log(f"\n측정 완료 {time.time()-t0:.0f}s")

In [ ]:
# CELL 5 — 【N0 무효 대조】 CI 기계가 없는 차이를 만들어내는가
# 하벽군을 무작위 반으로 갈라 두 반쪽의 비중 차를 잰다. 0을 포함해야 정상이다.
# ★ 통과는 '증거'가 아니다(검정력이 낮다). 실패만 경보로 읽는다.
NULL = {}
for pname, P in PANELS.items():
    g = P["pos"]
    if g not in RESULT[pname] or not RESULT[pname][g]["testable"]:
        run.log(f"\nN0 {pname}: 양성군 표본 부족 — 생략"); continue
    idx = MEMB[pname][g].copy()
    rs = np.random.RandomState(SEED0 + 7); rs.shuffle(idx)
    h = len(idx) // 2
    ci = CLASSES.index(P["cls"])
    mA = measure(ci, np.sort(idx[:h]))
    mB = measure(ci, np.sort(idx[h:]))
    d = mA["_bs"] - mB["_bs"]
    d = d[~np.isnan(d)]
    lo, hi = q(d, 2.5), q(d, 97.5)
    ok = bool(lo <= 0 <= hi)
    NULL[pname] = {"n_half": int(h), "diff": float(np.mean(d)), "ci": [lo, hi], "ok": ok}
    run.log(f"\n【N0】 {pname} · {g} 무작위 반분 (각 n≈{h})")
    run.log(f"  비중 차 = {np.mean(d):+.3f}  [{lo:+.3f}, {hi:+.3f}] → "
            + ("✅ 0 포함 (경보 없음)" if ok
               else "🚨 0을 제외 — CI가 너무 좁다. 아래 판정 전부 의심하라"))

In [ ]:
# CELL 6 — 사전등록 채점
MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def decide(lo, hi, thr, direction):
    """지지/기각/미결 3분. CI가 임계값을 걸치면 기각이 아니라 미결."""
    if direction == ">":
        if lo > thr: return True
        if hi < thr: return False
    else:
        if hi < thr: return True
        if lo > thr: return False
    return None

SCORE = {}
for pname, P in PANELS.items():
    pos, neg = P["pos"], P["neg"]
    R = RESULT.get(pname, {})
    run.log("\n" + "=" * 104)
    run.log(f"【{pname} 패널 사전등록 채점】")
    run.log("=" * 104)
    base = R.get("__전체__", {}).get("share")
    run.log(f"  기준선(전체) 비중 = " + ("정의불가" if base is None else f"{base:.1%}")
            + "   ← 같은 실행·같은 지표 (실험10″의 교훈)")

    usable = (pos in R and neg in R and R[pos]["testable"] and R[neg]["testable"]
              and R[pos]["share"] is not None and R[neg]["share"] is not None)
    if not usable:
        why = f"{pos} n={CENSUS[pname].get(pos,{}).get('순수','?')} · " \
              f"{neg} n={CENSUS[pname].get(neg,{}).get('순수','?')}"
        run.log(f"  P-A/P-B/P-C → ⚠️ 판정불가 (표본 또는 분모: {why})")
        SCORE[pname] = {"P-A": None, "P-B": None, "P-C": None, "usable": False}
        continue

    dif = R[pos]["_bs"] - R[neg]["_bs"]; dif = dif[~np.isnan(dif)]
    dlo, dhi = q(dif, 2.5), q(dif, 97.5)
    PA = decide(dlo, dhi, 0.0, ">")
    run.log(f"  P-A ★ 비중({pos}) − 비중({neg}) > 0 → {MARK[PA]}  "
            f"Δ={np.mean(dif):+.3f} [{dlo:+.3f}, {dhi:+.3f}]")
    run.log(f"        ({R[pos]['share']:.1%} vs {R[neg]['share']:.1%})")

    nlo, nhi = R[neg]["share_ci"]
    PB = decide(nlo, nhi, 0.33, "<")
    run.log(f"  P-B  비중({neg}) < 33% → {MARK[PB]}  "
            f"{R[neg]['share']:.1%} [{nlo:.0%}, {nhi:.0%}]")

    if base is None:
        PC = None
        run.log("  P-C  비중(양성군) > 기준선 → ⚠️ 미결 (기준선 정의불가)")
    else:
        bdif = R[pos]["_bs"] - R["__전체__"]["_bs"]; bdif = bdif[~np.isnan(bdif)]
        blo, bhi = q(bdif, 2.5), q(bdif, 97.5)
        PC = decide(blo, bhi, 0.0, ">")
        run.log(f"  P-C  비중({pos}) > 기준선 → {MARK[PC]}  "
                f"Δ={np.mean(bdif):+.3f} [{blo:+.3f}, {bhi:+.3f}]")
    SCORE[pname] = {"P-A": PA, "P-B": PB, "P-C": PC, "usable": True,
                    "share_pos": R[pos]["share"], "share_neg": R[neg]["share"],
                    "share_base": base, "diff_ci": [dlo, dhi]}

# ── 계측기 판정
mi = SCORE.get("MI", {})
sttc = SCORE.get("STTC", {})
n0_alarm = any(v["ok"] is False for v in NULL.values())

run.log("\n" + "=" * 104)
run.log("【계측기 판정】 유도군 지표는 해부학적 평면을 재는가")
run.log("=" * 104)
if n0_alarm:
    verdict = ("🚨 무효 — N0 무효 대조가 실패했다(같은 군을 반으로 갈랐는데 비중 차가 유의). "
               "CI가 너무 좁아 모든 판정이 신뢰할 수 없다. 부트스트랩 설계부터 고쳐야 한다")
elif mi.get("P-A") is True and sttc.get("P-A") is True:
    verdict = ("✅ 계측기 검증 통과(이중) — MI와 STTC 두 독립 패널에서 전두면 소견이 "
               "횡단면 소견보다 사지유도로 더 많이 열린다. 유도군 지표가 해부학적 평면을 "
               "재고 있다는 것이 '아는 답'으로 확인됐다 → 차별점 A의 표를 신뢰할 수 있다")
elif mi.get("P-A") is True:
    verdict = ("✅ 계측기 검증 통과(MI 단독) — 전두면 대 횡단면 대조가 예측대로 갈렸다. "
               "STTC 복제는 표본 또는 검정력에서 막혔다")
elif mi.get("P-A") is False:
    verdict = ("🚨 계측기 검증 실패 — 하벽경색이 전중격경색보다 사지유도를 **덜** 쓴다고 "
               "나왔다. 이건 해부학적으로 불가능하므로 지표가 평면을 재는 게 아니다. "
               "**실험10·10′·10″와 차별점 A를 전부 재검토 대상으로 내린다**")
else:
    verdict = ("⚠️ 미결 — 방향은 맞을 수 있으나 CI가 0을 걸친다. 계측기는 검증도 반증도 "
               "되지 않았다. 차별점 A는 '미검증 계측' 딱지를 달고 보류한다")
run.log(f"\n▶ {verdict}")
run.log("=" * 104)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(PANELS), figsize=(6.4 * len(PANELS), 3.9), squeeze=False)
for ax, (pname, P) in zip(axes[0], PANELS.items()):
    gs = [g for g in RESULT.get(pname, {}) if RESULT[pname][g]["share"] is not None]
    if not gs:
        ax.text(.5, .5, f"{pname}: 표본 부족", ha="center"); ax.axis("off"); continue
    v = [RESULT[pname][g]["share"] for g in gs]
    e = [[RESULT[pname][g]["share"] - RESULT[pname][g]["share_ci"][0] for g in gs],
         [RESULT[pname][g]["share_ci"][1] - RESULT[pname][g]["share"] for g in gs]]
    lab = ["기준선\n(전체)" if g == "__전체__" else g.replace("(", "\n(") for g in gs]
    ax.bar(range(len(gs)), v, yerr=e, capsize=4,
           color=["#888" if g == "__전체__" else "#c33" if g == P["neg"]
                  else "#36c" if g == P["pos"] else "#bbb" for g in gs])
    ax.axhline(0.33, ls="--", c="#c33", lw=1); ax.axhline(0, c="k", lw=.8)
    ax.set_xticks(range(len(gs))); ax.set_xticklabels(lab, fontsize=7)
    ax.set_ylabel("{I,II}가 가져가는 12유도 이득의 비중")
    ax.set_title(f"{pname} — 파랑=전두면 예상, 빨강=횡단면 예상", fontsize=9)
plt.tight_layout(); run.save_fig("finding_plane_share", fig); plt.show()

CLEAN = {p: {g: {k: v for k, v in m.items() if k != "_bs"} for g, m in R.items()}
         for p, R in RESULT.items()}
run.save_json("evaluation", {"census": CENSUS, "result": CLEAN, "score": SCORE,
                             "null_control": NULL, "verdict": verdict})

result = {"week": 2, "exp_id": "exp13_finding_plane", "quest": "ailab-2026-0015",
          "task": "유도군 지표가 해부학적 평면을 재는지 '아는 답'(하벽 vs 전중격)으로 검증",
          "split": "inter", "metric": "share_diff_frontal_minus_transverse_MI",
          "value": (None if not mi.get("usable")
                    else round(mi["share_pos"] - mi["share_neg"], 4)),
          "passed": bool(mi.get("P-A") is True and not n0_alarm),
          "date": time.strftime("%Y-%m-%d"), "training": "없음(실험10′ OOF 재분석)",
          "census": CENSUS, "subgroups": CLEAN, "score": SCORE, "null_control": NULL,
          "verdict": verdict,
          "summary": (("MI 하벽 " + f"{mi['share_pos']:.1%}" + " vs 전중격 "
                       + f"{mi['share_neg']:.1%}" + f" (기준선 {mi['share_base']:.1%})"
                       if mi.get("usable") else "MI 패널 판정불가")
                      + f" · P-A {MARK[mi.get('P-A')]} · " + verdict.split(' —')[0])}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp13_finding_plane_control.ipynb \\
      --quest ailab-2026-0015 --step "exp13-finding-plane-control" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")

---

## 결과 읽는 법

**P-A 하나만 보면 된다.** 나머지는 해석을 돕는 곁가지다.

| P-A | 뜻 | 퀘스트에 미치는 영향 |
|---|---|---|
| ✅ | 자가 평면을 잰다 | 차별점 A의 표가 **검증된 계측** 위에 선다. 실험10′의 혼합군 확증도 신뢰도가 올라간다. Cornell 해석은 여전히 미확증이지만 "지표 탓"은 배제된다 |
| ❌ | **자가 평면을 못 잰다** | 실험10·10′·10″가 무엇을 쟀는지 알 수 없다. 차별점 A 보류, 지표 설계부터 다시 |
| ⚠️ | 검정력 부족 | 차별점 A에 '미검증 계측' 딱지. 더 큰 데이터셋(Chapman/Ningbo)에서 재시도 |

**❌ 가 나오는 게 최악이 아니다.** 최악은 검증 없이 표를 논문에 싣는 것이다.

## 이 실험이 방법론에 남기는 것

1. **양성 대조(positive control)를 실험 큐에 정식으로 넣는 관행** — 지금까지 큐는 전부
   "모르는 것"이었다. 아는 것을 한 번 재보는 실험이 없으면 계측기가 언제 고장 났는지 모른다.
2. **같은 실행·같은 지표 기준선 행** — 실험10″에서 지표가 다른 두 수를 비교한 실수의 재발 방지.
3. **무효 대조(N0)** — 같은 집단을 반으로 갈라 "차이 없음"이 나오는지 확인. CI 기계 자체의 검산.

## 한계

- **모델은 subclass를 배운 적이 없다.** 5-superclass 분류기이므로 여기서 재는 것은
  "그 부위 경색에서 MI 확률이 유도구성에 따라 얼마나 오르나"이지 부위 진단 성능이 아니다.
  → 계측기 검증에는 이걸로 충분하지만, "하벽경색을 진단할 수 있다"로 읽으면 안 된다.
- **후벽경색은 '횡단면'으로 묶었지만 애매하다.** 후벽은 V7–V9를 안 붙이면 V1–V3의
  **상반변화**로 읽는다 — 횡단면 유도를 쓰긴 하나 직접 관측이 아니다. 예측에서 뺐고
  서술만 한다.
- **PTB-XL 라벨은 자동판독기 유래다.** 부위 라벨이 유도별 규칙으로 붙었다면
  순환논리 위험이 있다 — 다만 이번엔 그게 **오히려 검증에 유리**하다. 라벨이 유도 규칙을
  따랐다면 지표는 더 확실히 갈려야 하고, 그런데도 안 갈리면 지표가 확실히 고장 난 것이다.
